# 01 — Exploratory Data Analysis

This notebook explores the ingested game logs and engineered features.

**Prerequisites:** Run ingestion and feature building first:
```bash
python scripts/ingest_player_logs.py --players "Stephen Curry" "LeBron James"
python scripts/build_features.py
```

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from court_edge_agent.data.storage import load_game_logs, load_features
from court_edge_agent.config import settings

settings.ensure_dirs()

In [ ]:
# Load raw game logs
logs = load_game_logs(season=settings.default_season)
print(f"Game logs: {len(logs)} rows, {logs['player_id'].nunique()} players")
logs.head()

In [ ]:
# Load engineered features
features = load_features(season=settings.default_season)
print(f"Features: {len(features)} rows")
features.describe()

In [ ]:
# Stat distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, stat in zip(axes.flat, ['points', 'rebounds', 'assists', 'threes_made']):
    features[stat].dropna().hist(bins=30, ax=ax)
    ax.set_title(stat)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Rolling feature vs. actual — quick visual leakage check for one player
player_name = features['player_name'].iloc[0]
p = features[features['player_name'] == player_name].copy()
p = p.sort_values('game_date')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(p['game_date'], p['points'], label='Actual points', alpha=0.7)
ax.plot(p['game_date'], p['rolling_5_points'], label='Rolling 5 (pre-game)', alpha=0.7, linestyle='--')
ax.set_title(f'{player_name} — Points vs. Rolling-5 Feature')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()